In [1]:
import json
import os
import pandas as pd

In [2]:
with open('', 'r') as f:
    corr_dict = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: ''

In [3]:
#choose instance for page, first choose direct_link before sc_link, then choose max pred_conf 

from operator import itemgetter

for batch in corr_dict.keys():

    for page in corr_dict[batch].keys():

        if corr_dict[batch][page] is not None and corr_dict[batch][page][0]['det_prob'] is not None:

            pred_confs = []
            sc_confs = []
            
            for i, inst in enumerate(corr_dict[batch][page]):
                
                if len(inst['pred_links']) >= 1:
                    pred_confs.append((i, float(inst['pred_conf'])))
                elif len(inst['sc_links']) >=1:
                    sc_confs.append((i, float(inst['pred_conf'])))

            chosen_index = None

            if len(pred_confs) >= 1: #and max(pred_confs, key=itemgetter(1))[1] > 0.95
                chosen_index = max(pred_confs, key=itemgetter(1))[0]
            elif len(sc_confs) >= 1: #and max(sc_confs, key=itemgetter(1))[1] > 0.95
                chosen_index = max(sc_confs, key=itemgetter(1))[0]

            if chosen_index is not None:
                corr_dict[batch][page][chosen_index]['chosen'] = True
            
                
        









In [8]:
#correct backside

import copy

with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_chosen_epoch3.json', 'r') as f:
    flipside_corr_dict = json.load(f)

from operator import itemgetter

for batch in flipside_corr_dict.keys():

    for i, page in enumerate(flipside_corr_dict[batch].keys()):

        if flipside_corr_dict[batch][page] is not None and flipside_corr_dict[batch][page][0]['det_prob'] is not None:
            
            
            
            if (i + 1) % 2 == 1:
                for ind_f, inst in enumerate(flipside_corr_dict[batch][page]):
                    if 'chosen' in inst:
                        front_chosen = copy.deepcopy(inst)
                        ind_front_chosen = ind_f

            elif (i + 1) % 2 == 0:

                prev_page = str(int(page.split('_')[1]) - 1)
                prev_page = prev_page.zfill(8)
                prev_page = '_'.join([batch, prev_page])

                for ind_b, inst in enumerate(flipside_corr_dict[batch][page]):
                    if 'chosen' in inst:
                        back_chosen =copy.deepcopy(inst)
                        ind_back_chosen = ind_b

                
                if ind_back_chosen is None or ind_front_chosen is None:
                    ind_front_chosen = None
                    ind_back_chosen = None
                    
                    continue
                
                elif len(front_chosen['pred_links']) >= 1 and len(back_chosen['sc_links']) >= 1:
                    #flipside_corr_dict[batch][page] =flipside_corr_dict[batch][prev_page]

                    flipside_corr_dict[batch][page][ind_back_chosen]['chosen'] = False
            
                elif len(front_chosen['sc_links']) >= 1 and len(back_chosen['pred_links']) >= 1:
                    #flipside_corr_dict[batch][prev_page] = flipside_corr_dict[batch][page]

                    flipside_corr_dict[batch][prev_page][ind_front_chosen]['chosen'] = False
            
                elif float(front_chosen['pred_conf']) >= float(back_chosen['pred_conf']):
                    #flipside_corr_dict[batch][page] = flipside_corr_dict[batch][prev_page]

                    flipside_corr_dict[batch][page][ind_back_chosen]['chosen'] = False
                
                elif float(front_chosen['pred_conf']) < float(back_chosen['pred_conf']):
                    #flipside_corr_dict[batch][prev_page] = flipside_corr_dict[batch][page]

                    flipside_corr_dict[batch][prev_page][ind_front_chosen]['chosen'] = False
                

                ind_front_chosen = None
                ind_back_chosen = None
                   

                    
            

            

In [17]:
if 0:
    print('a')

In [9]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_flipside_chosen_epoch3.json', 'w', encoding='utf8') as f:
    j = json.dumps(flipside_corr_dict, indent = 4, ensure_ascii=False)
    f.write(str(j))